In [ ]:
import numpy as np
from fury import window, actor
from dipy.tracking import utils

from data import data, affine, labels, bvals, bvecs, vox_size
from env_main import UnifiedTractographyEnv
from agent import EuDXAgent, RandomTractographyAgent


In [2]:
env = UnifiedTractographyEnv(
    data=data, 
    affine=affine, 
    labels=labels, 
    bvals=bvals, 
    bvecs=bvecs, 
    fa_threshold=0.25,      # Stopping criteria: GFA < 0.25
    max_curvature_deg=45,   # Stopping criteria: Angle > 45 deg
    max_steps=500           # Stopping criteria: Length > 500
)

c:\Users\Samsung\Desktop\tractography\main\env_main.py:65: UserWarning: Pass ['bvecs'] as keyword args. From version 2.0.0 passing these as positional arguments will result in an error. 
  gtab = gradient_table(bvals, bvecs)


In [3]:
agent = EuDXAgent(env)

In [4]:
seed_indices = np.argwhere(labels == 2)

In [5]:
seed_indices

array([[38, 38, 34],
       [38, 41, 33],
       [38, 42, 33],
       ...,
       [40, 76, 34],
       [40, 76, 35],
       [40, 76, 36]], dtype=int64)

In [6]:
all_streamlines_world = []

In [10]:
seed_mask=(labels==2)
seeds= utils.seeds_from_mask(seed_mask, affine, density=vox_size)


In [11]:

# For demonstration, let's track the first 10 seeds
for i in range(10):
    seed_point = seeds[i]
    
    # The agent's generate_streamline method handles the RL loop:
    # reset -> get_obs -> predict action -> step -> repeat
    streamline = agent.generate_streamline(seed_point)
    
    if len(streamline) > 1:
        all_streamlines_world.append(streamline)

print(f"Tracking complete. Generated {len(all_streamlines_world)} streamlines.")



Error: Seed must be a python integer, actual type: <class 'numpy.ndarray'>

In [ ]:
# ==========================================
# 3. Visualization (FURY)
# ==========================================
scene = window.Scene()

# Render ROI Context
env.render_seeds_mask(scene)  # Red points for seed regions
env.render_wm_surface(scene, opacity=0.1)  # Transparent White Matter shell

# Render the Streamlines
# Use different colors to distinguish them if desired
if all_streamlines_world:
    streamline_actor = actor.line(
        all_streamlines_world, 
        colors=(0, 1, 0),  # Green for EuDX lines
        linewidth=2
    )
    scene.add(streamline_actor)

    # Highlight the first seed point specifically
    first_seed_world = all_streamlines_world[0][0]
    scene.add(actor.sphere(
        centers=[first_seed_world], 
        colors=(1, 1, 0), 
        radii=1.0
    ))

# Interaction
print("Opening 3D visualization...")
window.show(scene)